In [2]:
from langchain_core.documents import Document

## data ingetion pipeine

## docuument data structure


In [3]:
doc=Document(page_content="This is a test document",
              metadata={"author":"shreyas","source": "test","page":1}
              )
doc

Document(metadata={'author': 'shreyas', 'source': 'test', 'page': 1}, page_content='This is a test document')

In [4]:
## creating a simple text file
import os
os.makedirs("../data/text_files", exist_ok=True)


In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """}

for file_path, content in sample_texts.items():
    with open(file_path,"w",encoding="utf-8") as f:
        f.write(content)

In [6]:
from langchain_community.document_loaders import TextLoader

C:\Users\shreyas jagtap\AppData\Local\Temp\ipykernel_20108\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\shreyas\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


## directory loader

In [9]:
from langchain_community.document_loaders import DirectoryLoader

In [10]:
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", # this for pattern matching al txt files in dir
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=False
)
doc=dir_loader.load()
doc

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

## chunking 

In [11]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [12]:
dirload=DirectoryLoader(
    "../data/dashboard",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=False
)
doc=dirload.load()
doc

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260622212613', 'source': '..\\data\\dashboard\\amazon sales dashboard.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Power BI Desktop\ntotal orders\n73.63K\nAverage Purchase Amount\n₹ 643.64\nTotal sale\n47.39M\nAmazon Sales Dashboard\nSum of Amount by ship_state\nMAHARAS…\nKARNATAKA\nUTTAR PR…\nTELANGANA\nTAMIL NADU\nDELHI\nKERALA\nWEST BEN…\nANDHRA P…\nHARYANA\n8.0M\n6.2M\n4.2M\n4.0M\n3.9M\n2.6M\n2.3M\n2.2M\n1.9M\n1.7M\nTotal Qty Sold by Category\nT-sh… Shirt\nBlazzer\nTrousers\nPerfume\nWallet\nSocks\nShoes\n26K 25K\n8K 6K\n1K 1K 0K 0K\nSum of Qty by Status\nShipped 62K (93.5%)\nCancelled 3K (4.3%)\nShipped Cancelled Return\nCount of Cancelled order by ship_state\nMAHARAS…\nKARNATAKA\nUTTAR PRA…\nTELANGANA\nTAMIL\n NADU\nKERALA\nDELHI\nWEST BENG…\nANDHRA P…\nHARYANA\n1.6K\n1.25K\n0.89K 0.86K 0.83K\n0.62K 0.48K 0.47K 0.45K 0.32K\nCount of Status (groups) by Category\nT-shirt\nShi

## MAKING CHUNKING

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List,Dict,Any,Tuple
import chromadb
from chromadb.config import Settings
import uuid

In [ ]:
class Embedding_manager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
      try:
        print(f"loading the model name:{self.model_name}")
        self.model=SentenceTransformer(self.model_name)
        print(f"modelm loaded suuccessfully{self.model_name}")
      except Exception as e:
         print(f"the error occured while loading the model:{self.model_name} error:{e}")
         raise

    def generate_embedding(self,texts:List[str])-> np.ndarray:
       if not self.model:
          print("model not found")
       else:
          embeddings=self.model.encode(texts,show_progress_bar=True)
          print(f"Generates Embeddings of shape{embeddings.shape()}")
          return embeddings

embedd=Embedding_manager()
embedd

loading the model name:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3355.16it/s]


modelm loaded suuccessfullyall-MiniLM-L6-v2
